# BirdCLEF+ 2026 — Submission Notebook

**How to use:**
1. Train locally: `python train.py --all-folds`
2. Upload `outputs/fold*_best.pth` files to a private Kaggle Dataset named `birdclef2026-checkpoints`
3. Add that dataset to this notebook (+ the competition data)
4. Submit — Kaggle runs this notebook on the hidden test set

This notebook is **self-contained**: all model/transform code is inlined.

In [ ]:
# Install extra packages only if they are missing.
# Kaggle submission reruns may have internet disabled, so avoid unconditional pip.
import importlib.util
import subprocess
import sys

missing = [p for p in ['timm', 'librosa', 'soundfile'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

In [ ]:
import glob
import os
import random

import numpy as np
import pandas as pd
import soundfile as sf
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T
from PIL import Image
from scipy.signal import resample_poly
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm

print('torch:', torch.__version__)
print('timm:', timm.__version__)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

DATA_DIR      = '/kaggle/input/competitions/birdclef-2026'
CKPT_DIR      = '/kaggle/input/datasets/qqqqhl0/birdclef2026-checkpoints'
OUTPUT_DIR    = '/kaggle/working'

TAXONOMY_CSV          = f'{DATA_DIR}/taxonomy.csv'
SAMPLE_SUBMISSION_CSV = f'{DATA_DIR}/sample_submission.csv'
TEST_SOUNDSCAPES_DIR  = f'{DATA_DIR}/test_soundscapes'

# Audio
SAMPLE_RATE   = 32_000
CLIP_DURATION = 5.0
N_SAMPLES     = int(SAMPLE_RATE * CLIP_DURATION)

# Mel spectrogram
N_MELS      = 128
N_FFT       = 1024
HOP_LENGTH  = 320
FMIN        = 20.0
FMAX        = 16_000.0
TOP_DB      = 80.0

# Model
MODEL_NAME  = 'tf_efficientnet_b0_ns'  # B4 exceeds 90-min CPU limit
NUM_CLASSES = 234
IN_CHANNELS = 3
DROP_RATE   = 0.3
IMG_SIZE    = 256
PRETRAINED  = False   # load weights from checkpoint, not ImageNet

# Inference
INFER_BATCH_SIZE = 32

DEVICE = (
    torch.device('cuda') if torch.cuda.is_available()
    else torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cpu')
)
print('Device:', DEVICE)

In [ ]:
# ── Species info ──────────────────────────────────────────────────────────────

def build_species_info(taxonomy_csv_path):
    tax = pd.read_csv(taxonomy_csv_path)
    species_list = [str(x) for x in tax['primary_label'].tolist()]
    label_to_idx = {sp: i for i, sp in enumerate(species_list)}
    return {'species_list': species_list, 'label_to_idx': label_to_idx}

species_info = build_species_info(TAXONOMY_CSV)
SPECIES_LIST = species_info['species_list']
print(f'Species: {len(SPECIES_LIST)}')

In [ ]:
# ── Transforms ─────────────────────────────────────────────────────────────────
# Audio is loaded as a raw waveform in the Dataset.
# Mel spectrograms are computed in batch here (torchaudio C++ STFT >> librosa).

_mel_tf  = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
    n_mels=N_MELS, f_min=FMIN, f_max=FMAX, power=2.0,
)
_db_tf   = T.AmplitudeToDB(top_db=TOP_DB)
_IN_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_IN_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def batch_waveforms_to_mel(waveforms: torch.Tensor) -> torch.Tensor:
    """
    Convert a batch of waveforms to ImageNet-normalised mel images.

    waveforms : (B, N_SAMPLES) float32 CPU tensor
    returns   : (B, 3, IMG_SIZE, IMG_SIZE) float32 CPU tensor
    """
    mel = _mel_tf(waveforms)                            # (B, N_MELS, T')
    mel = _db_tf(mel)                                   # dB scale
    mel = ((mel + TOP_DB) / TOP_DB).clamp(0, 1)        # normalise [0, 1]
    mel = mel.unsqueeze(1)                              # (B, 1, N_MELS, T')
    mel = F.interpolate(mel, size=(IMG_SIZE, IMG_SIZE),
                        mode='bilinear', align_corners=False)
    mel = mel.repeat(1, 3, 1, 1)                       # (B, 3, H, W)
    mel = (mel - _IN_MEAN) / _IN_STD                   # ImageNet normalise
    return mel


def audio_transform_val(waveform: np.ndarray) -> np.ndarray:
    """Tile-pad / center-crop to exactly N_SAMPLES; amplitude-normalise."""
    waveform = np.squeeze(waveform).astype(np.float32)
    if waveform.ndim != 1 or len(waveform) == 0:
        return np.zeros(N_SAMPLES, dtype=np.float32)
    if len(waveform) < N_SAMPLES:
        repeats  = (N_SAMPLES // len(waveform)) + 1
        waveform = np.tile(waveform, repeats)
    start    = (len(waveform) - N_SAMPLES) // 2
    waveform = waveform[start : start + N_SAMPLES]
    waveform /= (np.max(np.abs(waveform)) + 1e-6)
    return waveform

In [ ]:
# ── Inference Dataset ──────────────────────────────────────────────────────────
# Uses sample_submission.csv as the authoritative list of windows to predict.
# This guarantees row_ids match exactly — scanning the directory can mismatch.
# Audio is loaded with soundfile.read (true random access, no full-file decode).
# Returns raw waveforms so mel computation can be batched in the inference loop.

class SoundscapeInferenceDataset(Dataset):
    def __init__(self, sample_submission_path, test_soundscapes_dir):
        sample_sub = pd.read_csv(sample_submission_path)
        self.test_dir = test_soundscapes_dir
        self.items = []   # (row_id, filepath, end_sec)
        for _, row in sample_sub.iterrows():
            parts   = str(row['row_id']).split('_')
            end_sec = int(parts[-1])
            stem    = '_'.join(parts[:-1])
            fpath   = os.path.join(test_soundscapes_dir, f'{stem}.ogg')
            self.items.append((row['row_id'], fpath, end_sec))
        found = sum(1 for _, fp, _ in self.items if os.path.exists(fp))
        print(f'Windows from sample_submission: {len(self.items):,}  |  files found: {found:,}')

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        row_id, fpath, end_sec = self.items[idx]
        start_sec  = max(end_sec - int(CLIP_DURATION), 0)
        target_len = N_SAMPLES
        try:
            info    = sf.info(fpath)
            file_sr = info.samplerate
            start_f = int(start_sec * file_sr)
            end_f   = min(int(end_sec   * file_sr), info.frames)
            y, _    = sf.read(fpath, start=start_f, stop=end_f,
                              dtype='float32', always_2d=False)
            if y.ndim == 2:
                y = y.mean(axis=1)
            if file_sr != SAMPLE_RATE:
                y = resample_poly(y, SAMPLE_RATE, file_sr).astype(np.float32)
        except Exception:
            y = np.zeros(target_len, dtype=np.float32)
        y = audio_transform_val(y)
        return torch.from_numpy(y), row_id

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────

class BirdModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            MODEL_NAME, pretrained=PRETRAINED, in_chans=IN_CHANNELS,
            num_classes=0, global_pool='avg', drop_rate=DROP_RATE,
        )
        self.classifier = nn.Linear(self.backbone.num_features, NUM_CLASSES)

    def forward(self, x):
        return self.classifier(self.backbone(x))


def load_model(ckpt_path):
    model = BirdModel()
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE)
    model.eval()
    snd_cmap = ckpt.get('snd_cmap', ckpt.get('val_cmap', '?'))
    clip_cmap = ckpt.get('clip_cmap', '?')
    print(f'  {os.path.basename(ckpt_path)}  snd_cmAP={snd_cmap}  clip_cmAP={clip_cmap}')
    return model

In [ ]:
# ── Load checkpoints ──────────────────────────────────────────────────────────

ckpt_paths = sorted(glob.glob(os.path.join(CKPT_DIR, 'fold*_best.pth')))
if not ckpt_paths:
    raise FileNotFoundError(
        f'No checkpoints found in {CKPT_DIR}. '
        'Upload your trained weights as a Kaggle Dataset named "birdclef2026-checkpoints".'
    )

print(f'Loading {len(ckpt_paths)} checkpoint(s):')
models = [load_model(p) for p in ckpt_paths]

In [ ]:
# ── Inference ──────────────────────────────────────────────────────────────────
# soundfile random-access means each __getitem__ is independent →
# num_workers > 0 is safe and overlaps audio I/O with model compute.

dataset = SoundscapeInferenceDataset(SAMPLE_SUBMISSION_CSV, TEST_SOUNDSCAPES_DIR)
loader  = DataLoader(
    dataset,
    batch_size=INFER_BATCH_SIZE,
    shuffle=False,
    num_workers=2,      # safe: no shared file cache
    pin_memory=(DEVICE.type == 'cuda'),
)
print(f'Total 5-second windows: {len(dataset)}')

all_row_ids = []
all_probs   = []

with torch.no_grad():
    for waveforms, row_ids in tqdm(loader, desc='Inference'):
        # Batch mel on CPU (torchaudio C++ STFT, much faster than per-sample librosa)
        images = batch_waveforms_to_mel(waveforms).to(DEVICE)
        batch_probs = []
        for m in models:
            logits = m(images)
            batch_probs.append(torch.sigmoid(logits).cpu().float().numpy())
        avg_probs = np.mean(batch_probs, axis=0)
        all_row_ids.extend(row_ids)
        all_probs.append(avg_probs)

if all_probs:
    all_probs = np.concatenate(all_probs, axis=0)
    print(f'Predictions shape: {all_probs.shape}')
else:
    all_probs = None
    print('WARNING: no windows processed (test_soundscapes/ is empty).')
    print('This is expected in interactive sessions. Kaggle injects real')
    print('test files only during the hidden scoring rerun (Submit).')

In [ ]:
# ── Build and save submission ──────────────────────────────────────────────────
sample = pd.read_csv(SAMPLE_SUBMISSION_CSV)

if all_probs is not None:
    pred_df = pd.DataFrame(all_probs, columns=SPECIES_LIST)
    pred_df.insert(0, 'row_id', all_row_ids)

    # Left-join from sample_submission: guarantees exact rows & order.
    # fillna(0) covers any windows the model didn't process.
    sub = sample[['row_id']].merge(pred_df, on='row_id', how='left')
    sub[SPECIES_LIST] = sub[SPECIES_LIST].fillna(0.0)
else:
    sub = sample.copy()
    print('Using sample_submission as placeholder (no test files found).')

# Sanity checks — these would catch mismatches before Kaggle rejects the file
assert len(sub) == len(sample),                  f'Row mismatch: {len(sub)} vs {len(sample)}'
assert list(sub.columns) == list(sample.columns), 'Column mismatch'
assert sub[SPECIES_LIST].isnull().sum().sum() == 0, 'Null values found in predictions'
print(f'Sanity checks passed ✓  shape={sub.shape}')

out_path = os.path.join(OUTPUT_DIR, 'submission.csv')
sub.to_csv(out_path, index=False)
print(f'Saved → {out_path}')
sub.head()